# Stability of the final hyper-posterior draw

This notebook analyses the deployment-time variability of the disintegrated majority vote. For each dataset and method, the hyper-posterior is learned **once**, the best cross-bounding vote is selected **once**, and 100 independent majority-vote weights are sampled from the selected hyper-posterior.

The reported variability is therefore conditional on one learned hyper-posterior. It does not measure variability across data splits or training seeds. The Dirichlet-mean vote is deterministic conditional on the learned hyper-posterior. Its reported bound value is the bound expression evaluated at the Dirichlet mean; it is not claimed as a disintegrated PAC-Bayes certificate for that deterministic point.

In [ ]:
from pathlib import Path
from matplotlib.lines import Line2D

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Code, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "Analysis":
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_ROOT = PROJECT_ROOT / "results_variability"
EXPORT_ROOT = PROJECT_ROOT / "Analysis" / "variability_exports"
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

METHOD_LABELS = {
    "des_dir": "DIS-V",
    "des_rivasplata_dir": "DIS-R",
}
METHOD_COLORS = {
    "des_dir": "#c738c3",
    "des_rivasplata_dir": "#37c842",
}
PREDICTOR_LABELS = {
    "stumps-uniform": "Decision stumps",
    "rf": "Random forests",
}
DATASET_ORDER = [
    "MUSH", "HABER", "SVMGUIDE", "TTT", "CODRNA",
    "PHIS", "ADULT", "SPLICE", "AUSTRALIAN",
    "SENSORLESS", "PROTEIN", "PENDIGITS", "SHUTTLE",
    "MNIST", "FASHION",
]

# Central font-size controls for every figure in the notebook.
FONT_SIZES = {
    "base": 8.5,
    "axis_label": 11,
    "title": 11,
    "legend": 9,
    "x_tick": 11,
    "y_tick": 11,
}

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["STIXGeneral", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": FONT_SIZES["base"],
    "axes.labelsize": FONT_SIZES["axis_label"],
    "axes.titlesize": FONT_SIZES["title"],
    "legend.fontsize": FONT_SIZES["legend"],
    "xtick.labelsize": FONT_SIZES["x_tick"],
    "ytick.labelsize": FONT_SIZES["y_tick"],
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

In [ ]:
def nested(config, section, key, default=None):
    return config.get(section, {}).get(key, default)


def load_results(root):
    summary_rows, draw_rows, skipped = [], [], []
    files = sorted(root.rglob("final-draw-variability.npy"))

    for path in files:
        try:
            result = np.load(path, allow_pickle=True).item()
        except Exception as error:
            skipped.append((path, str(error)))
            continue

        config = result.get("config", {})
        risk = nested(config, "training", "risk")
        predictor = nested(config, "model", "pred")
        error_q = np.asarray(result["draw-test-error-quantiles"], dtype=float)
        bound_q = np.asarray(result["draw-bound-quantiles"], dtype=float)
        config_id = str(path.relative_to(root).parent)
        base = {
            "config_id": config_id,
            "dataset": config.get("dataset"),
            "method": risk,
            "method_label": METHOD_LABELS.get(risk, risk),
            "predictor": predictor,
            "predictor_label": PREDICTOR_LABELS.get(predictor, predictor),
            "bound_type": nested(config, "bound", "type"),
            "prior": nested(config, "model", "prior"),
            "order": nested(config, "training", "order"),
            "learnt_prior": nested(config, "pretraining", "learnt_prior"),
            "seed": nested(config, "training", "seed"),
            "selected_vote": result.get("selected-vote"),
            "num_draws": int(result["num-draws"]),
            "result_file": str(path),
        }
        summary_rows.append({
            **base,
            "error_mean": float(result["draw-test-error-mean"]),
            "error_variance": float(result["draw-test-error-variance"]),
            "error_std": float(result["draw-test-error-std"]),
            "error_q05": error_q[0],
            "error_median": error_q[1],
            "error_q95": error_q[2],
            "mean_vote_error": float(result["mean-vote-test-error"]),
            "bound_mean": float(result["draw-bound-mean"]),
            "bound_variance": float(result["draw-bound-variance"]),
            "bound_std": float(result["draw-bound-std"]),
            "bound_q05": bound_q[0],
            "bound_median": bound_q[1],
            "bound_q95": bound_q[2],
            "mean_vote_bound": float(result["mean-vote-bound-value"]),
            "training_time": float(result["training-time"]),
        })

        errors = np.asarray(result["draw-test-errors"], dtype=float)
        bounds = np.asarray(result["draw-bounds"], dtype=float)
        if len(errors) != len(bounds):
            raise ValueError(f"Mismatched draw arrays in {path}")
        for index, (error, bound) in enumerate(zip(errors, bounds)):
            draw_rows.append({
                **base, "draw": index, "test_error": error, "bound": bound
            })

    summary = pd.DataFrame(summary_rows)
    draws = pd.DataFrame(draw_rows)
    return summary, draws, files, skipped


summary_df, draws_df, result_files, skipped_files = load_results(RESULTS_ROOT)
print(f"Loaded {len(summary_df)} completed configurations and {len(draws_df)} final draws.")
if skipped_files:
    print(f"Skipped {len(skipped_files)} unreadable files (possibly still being written).")
if summary_df.empty:
    print("No results found. Run ./variability.sh before executing this notebook.")

In [ ]:
if not summary_df.empty:
    duplicates = summary_df.duplicated(["dataset", "method", "predictor"], keep=False)
    if duplicates.any():
        print("Warning: several hyperparameter configurations exist for some dataset/method pairs.")
        display(summary_df.loc[duplicates, ["dataset", "method", "prior", "order", "learnt_prior"]])

    coverage = (
        summary_df.groupby(["predictor_label", "method_label"], observed=True)
        .agg(configurations=("dataset", "size"), datasets=("dataset", "nunique"))
        .reset_index()
    )
    display(coverage)

## Final-draw performance and bound stability

Circles show the mean over 100 independent final draws, capped horizontal segments show the empirical 5th–95th percentile range, and diamonds optionally show the deterministic Dirichlet-mean vote. Decision stumps and random forests are exported as independent figures so that they can be assembled in LaTeX.

In [ ]:
def ordered(values):
    values = list(dict.fromkeys(values))
    known = [name for name in DATASET_ORDER if name in values]
    return known + sorted(set(values) - set(known))


def save_figure(fig, stem):
    fig.savefig(EXPORT_ROOT / f"{stem}.pdf")
    fig.savefig(EXPORT_ROOT / f"{stem}.png", dpi=300)


PREDICTOR_EXPORT_NAMES = {
    "stumps-uniform": "decision_stumps",
    "rf": "random_forests",
}

# Central controls for the four standalone figures. Set show_mean_vote=False
# to remove every diamond, and show_legend=False when legends are handled in LaTeX.
INTERVAL_FIGURE_OPTIONS = {
    "show_mean_vote": False,
    "mean_vote_y_offset": 0.07,
    "interval_cap_height": 0.045,
    "left_padding_fraction": 0.035,
    "show_title": False,
    "show_legend": True,
    "legend_inside": False,
    "figsize": (5.2, 4.25),
}

# Legend controls for the inside and outside modes.
INTERVAL_LEGEND_OPTIONS = {
    "inside": {
        "loc": "best",
        "ncol": 1,
        "frameon": True,
        "framealpha": 0.9,
        "borderaxespad": 0.6,
        "handlelength": 2.0,
    },
    "outside": {
        "loc": "lower center",
        "bbox_to_anchor": (0.5, 1.02),
        "ncol": 1,
        "frameon": False,
        "handlelength": 2.2,
    },
}

# Number of decimals used in the LaTeX tables associated with the figures.
INTERVAL_TABLE_DIGITS = {"error": 3, "bound": 4}


def interval_figure(
    data, metric, predictor, filename, *, show_mean_vote=True,
    mean_vote_y_offset=0.07, interval_cap_height=0.045, show_title=True,
    left_padding_fraction=0.035, show_legend=False, legend_inside=True,
    legend_kwargs=None, figsize=(5.2, 4.25),
):
    panel_data = data[data["predictor"] == predictor]
    methods = [m for m in METHOD_LABELS if m in set(panel_data["method"])]
    if panel_data.empty or not methods:
        return None

    fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)
    offsets = dict(zip(methods, np.linspace(-0.16, 0.16, len(methods))))
    scale = 100.0 if metric == "error" else 1.0
    xlabel = "Test error (%)" if metric == "error" else "PAC-Bayes bound value"
    datasets = ordered(panel_data["dataset"])
    positions = {dataset: i for i, dataset in enumerate(datasets)}

    for method in methods:
        method_data = panel_data[panel_data["method"] == method]
        color = METHOD_COLORS[method]
        for _, row in method_data.iterrows():
            y = positions[row["dataset"]] + offsets[method]
            low = row[f"{metric}_q05"] * scale
            high = row[f"{metric}_q95"] * scale
            mean = row[f"{metric}_mean"] * scale
            # Draw quantiles independently from the mean: for discrete samples,
            # the empirical mean can occasionally lie outside the 5–95% interval.
            ax.plot([low, high], [y, y], color=color, linewidth=1.6, zorder=3)
            ax.vlines(
                [low, high], y - interval_cap_height, y + interval_cap_height,
                color=color, linewidth=1.1, zorder=3,
            )
            ax.scatter(mean, y, s=17, color=color, marker="o", zorder=4)

            if show_mean_vote:
                fixed = row[f"mean_vote_{metric}"] * scale
                fixed_y = y + mean_vote_y_offset
                ax.plot(
                    [mean, fixed], [y, fixed_y], color=color, linewidth=0.65,
                    alpha=0.45, zorder=2,
                )
                ax.scatter(
                    fixed, fixed_y, s=27, facecolor="white", edgecolor=color,
                    linewidth=1.15, marker="D", zorder=4,
                )

    ax.set_yticks(range(len(datasets)), datasets)
    ax.invert_yaxis()
    ax.set_xlabel(xlabel)
    # Keep zero as the first visible tick while leaving room for near-zero markers.
    _, right = ax.get_xlim()
    ax.set_xlim(left=-left_padding_fraction * right)
    if show_title:
        ax.set_title(PREDICTOR_LABELS[predictor], loc="left")
    ax.grid(axis="x", color="0.9", linewidth=0.6)
    ax.grid(axis="y", visible=False)

    if show_legend:
        method_handles = [
            Line2D([0], [0], color=METHOD_COLORS[m], lw=2, label=METHOD_LABELS[m])
            for m in methods
        ]
        semantic_handles = [
            Line2D(
                [0], [0], color="0.25", marker="o", lw=1.5,
                label="Draw mean; empirical 5–95% range",
            ),
        ]
        if show_mean_vote:
            semantic_handles.append(
                Line2D(
                    [0], [0], color="0.25", marker="D", markerfacecolor="white",
                    lw=0, label="Dirichlet-mean vote",
                )
            )
        mode = "inside" if legend_inside else "outside"
        options = dict(INTERVAL_LEGEND_OPTIONS[mode])
        options.update(legend_kwargs or {})
        legend_target = ax if legend_inside else fig
        legend_target.legend(handles=method_handles + semantic_handles, **options)

    save_figure(fig, filename)
    return fig


def export_interval_latex_table(
    data, metric, predictor, filename, *, include_mean_vote=False, digits=None,
):
    panel_data = data[data["predictor"] == predictor]
    methods = [m for m in METHOD_LABELS if m in set(panel_data["method"])]
    datasets = ordered(panel_data["dataset"])
    scale = 100.0 if metric == "error" else 1.0
    digits = INTERVAL_TABLE_DIGITS[metric] if digits is None else digits
    unit = r" (\%)" if metric == "error" else ""

    rows = []
    for dataset in datasets:
        for method in methods:
            selected = panel_data[
                (panel_data["dataset"] == dataset)
                & (panel_data["method"] == method)
            ]
            for _, row in selected.iterrows():
                table_row = {
                    "Dataset": dataset,
                    "Method": METHOD_LABELS[method],
                    f"Draw mean{unit}": f"{scale * row[f'{metric}_mean']:.{digits}f}",
                    rf"$q_{{0.05}}${unit}": f"{scale * row[f'{metric}_q05']:.{digits}f}",
                    rf"$q_{{0.95}}${unit}": f"{scale * row[f'{metric}_q95']:.{digits}f}",
                }
                if include_mean_vote:
                    label = (
                        f"Dirichlet-mean error{unit}" if metric == "error"
                        else "Expression at Dirichlet mean"
                    )
                    table_row[label] = (
                        f"{scale * row[f'mean_vote_{metric}']:.{digits}f}"
                    )
                rows.append(table_row)

    table = pd.DataFrame(rows)
    column_format = "ll" + "r" * (len(table.columns) - 2)
    latex = table.to_latex(
        index=False, escape=False, column_format=column_format,
    )
    output_path = EXPORT_ROOT / f"{filename}.tex"
    output_path.write_text(latex, encoding="utf-8")
    return table, latex

In [ ]:
if not summary_df.empty:
    error_figures = {}
    error_tables = {}
    for predictor, export_name in PREDICTOR_EXPORT_NAMES.items():
        figure_stem = f"final_draw_test_error_{export_name}"
        fig = interval_figure(
            summary_df, "error", predictor,
            figure_stem,
            **INTERVAL_FIGURE_OPTIONS,
        )
        if fig is not None:
            error_figures[predictor] = fig
            error_tables[predictor], table_latex = export_interval_latex_table(
                summary_df, "error", predictor, f"{figure_stem}_table",
                include_mean_vote=INTERVAL_FIGURE_OPTIONS["show_mean_vote"],
            )
            plt.show()
            display(Code(table_latex, language="latex"))

In [ ]:
if not summary_df.empty:
    bound_figures = {}
    bound_tables = {}
    for predictor, export_name in PREDICTOR_EXPORT_NAMES.items():
        figure_stem = f"final_draw_bound_{export_name}"
        fig = interval_figure(
            summary_df, "bound", predictor,
            figure_stem,
            **INTERVAL_FIGURE_OPTIONS,
        )
        if fig is not None:
            bound_figures[predictor] = fig
            bound_tables[predictor], table_latex = export_interval_latex_table(
                summary_df, "bound", predictor, f"{figure_stem}_table",
                include_mean_vote=INTERVAL_FIGURE_OPTIONS["show_mean_vote"],
            )
            plt.show()
            display(Code(table_latex, language="latex"))